# What's in a Word: Breaking Texts and Making Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-Madison-Digital-Scholarship-Hub/text-and-data/blob/main/03-Tokenization/Whats-in-a-Word.ipynb)

Before text can be analyzed computationally, it has to be taken apart—split into pieces, normalized, stripped of noise. 

In this notebook, we'll introduce the preprocessing behind nearly all text analysis: tokenization, lemmatization, stemming, and stop word removal. We'll also distinguish this sense of "token" from how the word is used in large language models, a difference that reveals a lot about both. Our goal is to understand not just how to prepare text, but what each choice quietly decides.

> **Important Step**

Run the cells in order from top to bottom. Each cell depends on the ones above it. You don't need a GPU for this notebook; Colab's default CPU runtime is fine.

In [ ]:
""" If you're working in Colab, run this cell to bring everything, including the
saved copies of today's books, over from the GitHub repository. It's safe to re-run."""
import os # lets us check which folder we're in
if not os.path.exists('Whats-in-a-Word.ipynb'): # already in the right folder? then there's nothing to do
    if not os.path.exists('text-and-data'): # only download the repository once
        !git clone https://github.com/UW-Madison-Digital-Scholarship-Hub/text-and-data.git
    %cd text-and-data/03-Tokenization

## Packages
We're going to work with a set of Python packages for helping us access and manipulate textual data, including: 
- [**nltk**](https://www.nltk.org/): The Natural Language Toolkit, one of the most common packages for learning natural language processing
- [**spaCy**](https://spacy.io/): A faster, production oriented package for NLP tasks
- [**pandas**](https://pandas.pydata.org/): An open source data manipulation and analysis package
- [**gutenbergpy**](https://pypi.org/project/gutenbergpy/): A library for filtering, downloading, and querying Project Gutenberg data

In [ ]:
# Packages to install if necessary
# nltk, spaCy, and pandas come preinstalled in Colab, but gutenbergpy doesn't
%pip install -q nltk spacy pandas gutenbergpy

# spaCy also needs a trained English model, so we download it if it isn't already installed
import spacy
if not spacy.util.is_package("en_core_web_sm"):
    spacy.cli.download("en_core_web_sm")
import nltk

## What's a Word?

There are many different computational text analysis methods but most of them begin at the level of the word or, in the parlance of Natural Language Processing (NLP), the **token**. From a computational perspective, words are sequences of characters, which are themselves represented by bits of codes. 

The combination of these bits (or, rather bytes) of code has no inherent meaning from the perspective of the computer, but by manipulating, counting, and analyzing these codes we can explore and analyze patterns in ways that reveal underlying trends, extrapolate meaning, and explore characteristics of texts that range from style and sentiment to the portrayal of gender and topical trends within textual datasets. 

There are many facets to these analytical methods but one thing that they all have in common is that, like Genesis, they begin with the Word. 

## Tokenization

The first step in our process is going to be breaking up strings of text into the component elements or **tokens**. Depending on our particular approach, these tokens can include only whole words or they might include punctuation or reduced or expanded versions of words (e.g., we may reduce "buses" to "bus" or expand contractions like "don't" into "do not"). 

We will generally start with text stored as a `string`, a type of data format in Python where alphanumeric characters and spaces are combined into a single entity. A string could be something like `'It was the best of times, it was the worst of times'` or it could be the entire text of *A Tale of Two Cities*. 

Tokenization involves taking a string of text and reducing it to a `list` (another data format) of `strings`. One larger chunk of data becomes multiple smaller chunks of data that are easier to manipulate and perform at least some calculations on. 

We'll start the tokenization with a Python library known as the *Natural Language Toolkit* or `nltk`. 

In [ ]:
# Import the packages and modules needed for tokenization
import nltk # import the Natural Language Toolkit
nltk.download('punkt_tab') # import the tokenization dataset from NLTK
from nltk.tokenize import word_tokenize # import the word tokenization module

In [44]:
open0 = 'It was the best of times, it was the worst of times.'
tokens0 = word_tokenize(open0)
print(tokens0)

['It', 'was', 'the', 'best', 'of', 'times', ',', 'it', 'was', 'the', 'worst', 'of', 'times', '.']


In [45]:
open1 = 'Call me Ishmael.'
tokens1 = word_tokenize(open1)
print(tokens1)

['Call', 'me', 'Ishmael', '.']


## Stemming and Lemmatization

Stemming and Lemmatization are Natural Language Processing techniques for reducing derived or inflected words into a common base form for tasks such as counting. 

**Stemming** uses some basic heuristics to reduce common word forms to their root form or "stem." Note that not all stems are valid word forms (as we'll see). 
**Lemmatization** uses more sophisticated rules to return a valid dictionary form of a root word or "lemma."

For instance, imagine a sentence like "He thought the Lakeshore Path was his favorite run of all the runs he had done this year." 

We have the words "run" and "runs" as the singular and plural forms of the same noun. If we were counting words, we'd likely want to count them as versions of the same word. This is where stemming and lemmatization come in as processes to simplify word forms.

Let's look at an example.

In [49]:
sent0 = 'The running man thought the Lakeshore Path was his favorite run of all the runs he had done this year'
sent_tokens0 = word_tokenize(sent0) #tokenize the sentence into a list of tokens

from nltk.stem import PorterStemmer
porter = PorterStemmer()

for token in sent_tokens0:
    print(porter.stem(token))

the
run
man
thought
the
lakeshor
path
wa
hi
favorit
run
of
all
the
run
he
had
done
thi
year


We can see that this correctly simplified the nouns "run" and "runs" into the common stem of "run" but it has also reduced "running" (which, in "the running man," describes the man) to "run," which may or may not be what we want. Additionally, it cuts off some other words in ways that are usable but somewhat crude.

Stemming is essentially a quick and dirty shortening technique. It requires less processing power and time to run but the results are less nuanced than lemmatization. 

Let's take a look at how lemmatization handles this sentence: 

In [52]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

print(lemmatizer.lemmatize('running'))

[nltk_data] Downloading package wordnet to /Users/bkrien/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


running


In [53]:
for token in sent_tokens0:
    print(lemmatizer.lemmatize(token))

The
running
man
thought
the
Lakeshore
Path
wa
his
favorite
run
of
all
the
run
he
had
done
this
year


The lemmatizer leaves more words intact than the stemmer did ("his" stays "his," "this" stays "this," "Lakeshore" stays "Lakeshore"), but look closely at "was": it became "wa," the same mistake the stemmer made. Why? Without any other information, the WordNet lemmatizer assumes every word is a noun, so it reads "was" as the plural of a (nonexistent) noun "wa." For the same reason, it leaves "running," "thought," "had," and "done" untouched, since none of them look like plural nouns.

The lemmatizer needs to know what *kind* of word it's looking at, which is exactly what part-of-speech tagging gives us. There are reasons to use a stemmer over a lemmatizer in specific contexts such as search/information retrieval but, in general, for most computational analysis in English and other languages for which a lemmatizing package exists, the lemmatizer (with part-of-speech information) tends to be the better choice. 

## Part-of-Speech (POS) Tagging

When analyzing texts it can be useful to know what part of speech (e.g., noun, adjective, verb, etc.) a given token belongs to. This can help with disambiguating between different words that have the same spelling. For instance, we can have the word "count" as a: 
- Verb, as in "I'm going to count the number of apples." 
- Noun as a title, as in "Count Dracula" or the "Count" on Sesame Street
- Noun as a record of the process of counting, as in "the count was 0 and 1" in baseball

Understanding what part of speech a word takes is highly contextual and relies on its position in a sentence and relative to other words. We can use a part-of-speech tagging algorithm to categorize words based on their part of speech. 

NLTK's tagger uses the tags from the [Penn Treebank](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html). Here are some of the most common ones:

| Tag | Part of Speech | Examples |
|-----|----------------|----------|
| NN | Noun, singular or mass | fox, dog, table |
| NNP | Proper noun, singular | John, London, Apple |
| VBZ | Verb, 3rd person singular present | jumps, is |
| VBD | Verb, past tense | had, went, walked |
| JJ | Adjective | quick, brown, lazy |
| RB | Adverb | completely, quickly, not |
| DT | Determiner | the, a, that |
| IN | Preposition or subordinating conjunction | over, for, in |

In [56]:
nltk.download('averaged_perceptron_tagger_eng') # download the tagging algorithm dataset

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/bkrien/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
tagged_sent0 = nltk.pos_tag(sent_tokens0) # assign tags to each of the tokens in the sentence
print(tagged_sent0)

[('The', 'DT'), ('running', 'VBG'), ('man', 'NN'), ('thought', 'VBD'), ('the', 'DT'), ('Lakeshore', 'NNP'), ('Path', 'NNP'), ('was', 'VBD'), ('his', 'PRP$'), ('favorite', 'JJ'), ('run', 'NN'), ('of', 'IN'), ('all', 'PDT'), ('the', 'DT'), ('runs', 'NNS'), ('he', 'PRP'), ('had', 'VBD'), ('done', 'VBN'), ('this', 'DT'), ('year', 'NN')]


### Lemmatizing with POS Tags
The NLTK lemmatizer does not require a part of speech tag in order to operate but having the POS tag makes the lemmatizer more accurate because POS tags provide useful context for disambiguating what the appropriate lemma of a given word in a given context should be. 

For instance, if we run the lemmatizer over the tagged parts of speech list, we can pass each token's tag along with it. There's one wrinkle: the WordNet lemmatizer doesn't understand Penn Treebank tags like `VBD`. It only knows four parts of speech (noun, verb, adjective, and adverb), so we first need to translate each tag into one of those.

In [65]:
# WordNet only knows four parts of speech, so we map the Penn Treebank tags onto them
def penn_to_wordnet(tag):
    """Convert a Penn Treebank tag (e.g., 'VBD') to the WordNet code the lemmatizer expects."""
    if tag.startswith('J'):
        return 'a' # adjective
    elif tag.startswith('V'):
        return 'v' # verb
    elif tag.startswith('R'):
        return 'r' # adverb
    else:
        return 'n' # noun (WordNet's default)

for token, tag in tagged_sent0:
    print(token, '->', lemmatizer.lemmatize(token, pos=penn_to_wordnet(tag)))


The -> The
running -> run
man -> man
thought -> think
the -> the
Lakeshore -> Lakeshore
Path -> Path
was -> be
his -> his
favorite -> favorite
run -> run
of -> of
all -> all
the -> the
runs -> run
he -> he
had -> have
done -> do
this -> this
year -> year


We can see that this is a different output from the previous lemmatizer. Now that the lemmatizer knows which words are verbs, "was," "thought," "had," and "done" are converted to their base (dictionary) forms: "be," "think," "have," and "do." 

"Running" also becomes "run," just like it did with the stemmer. The tagger labeled it `VBG` (a verb form ending in "-ing"), so the lemmatizer treated it as a verb, even though in "the running man" it describes the man the way an adjective would. Whether "run" is the right result depends on what you're trying to count.

One of the key takeaways here is that the order of cleaning or processing text can really matter. The lemmatizer works better when it has part-of-speech tags, and the tagger works best on the full, original sentence, so we tag *before* we lemmatize, and we do both *before* removing stop words or anything else that changes the sentence. Different orders of operation are going to lend themselves to different kinds of analysis and understanding how the pieces fit together is important. 

## Stop Words

Not all tokens are equally valuable for all forms of analysis. While nouns tend to tell us what a text is about and verbs tend to tell us what's happening in a text, some types of words carry relatively little meaning. We call words that are extremely common but which carry little semantic meaning **stop words** and eliminating them is a common strategy in cleaning up text for analysis. While there is no single comprehensive list of stopwords for any language or dialect, they commonly include articles (such as "a" and "the"), prepositions (such as "in" and "on"), and pronouns (such as "she", "he", and "they"). 

Removing these stop words can help reduce the "noise" in a given text but the choice of stop words depends on context. For instance, in completing an exploration of gendered language on an English language text, keeping pronouns might be critical to the analysis that is being undertaken. 

We can take a look at the removal of stopwords from the opening paragraph of *A Tale of Two Cities*:

In [75]:
open_para = ' It was the best of times, it was the worst of times, it was the age of wisdom, it was the age of foolishness, it was the epoch of belief, it was the epoch of incredulity, it was the season of Light, it was the season of Darkness, it was the spring of hope, it was the winter of despair, we had everything before us, we had nothing before us, we were all going direct to Heaven, we were all going direct the other way—in short, the period was so far like the present period, that some of its noisiest authorities insisted on its being received, for good or for evil, in the superlative degree of comparison only.'

In [78]:
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/bkrien/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [81]:
stop_words = set(stopwords.words('english')) # Select the English list of stopwords
tokens = word_tokenize(open_para)
filtered_tokens = [token for token in tokens if token.lower() not in stop_words]

print(filtered_tokens)

['best', 'times', ',', 'worst', 'times', ',', 'age', 'wisdom', ',', 'age', 'foolishness', ',', 'epoch', 'belief', ',', 'epoch', 'incredulity', ',', 'season', 'Light', ',', 'season', 'Darkness', ',', 'spring', 'hope', ',', 'winter', 'despair', ',', 'everything', 'us', ',', 'nothing', 'us', ',', 'going', 'direct', 'Heaven', ',', 'going', 'direct', 'way', '—', 'short', ',', 'period', 'far', 'like', 'present', 'period', ',', 'noisiest', 'authorities', 'insisted', 'received', ',', 'good', 'evil', ',', 'superlative', 'degree', 'comparison', '.']


This is not a particularly readable list of tokens, but it does foreground some of the more meaning-laden words from the paragraph. It lends itself to a different kind of reading and analysis than the full text version and the utility of removing stopwords and the other techniques that we've covered begin to reveal their analytic potential when we move to looking at longer examples.

## Book-Length Example

Looking at the level of a sentence, it's clear that stop words carry relatively little meaning (in most contexts, at least). But what about at the level of a book? Let's take a look at a few examples. 

We'll use the `gutenbergpy` library, a library for querying and downloading books directly from Project Gutenberg. It allows us to pull the text of books from Project Gutenberg using the book's ID number, which can be found at the end of the URL for a specific book (e.g., the URL for one version of *Moby-Dick* is `https://www.gutenberg.org/ebooks/2701` and its ID number is `2701`).

One of the nice things about this particular package is that it allows you to download the text of the book without the Project Gutenberg headings and additional information, which saves a cleaning step. 

In [ ]:
# Let's start by getting a book from Project Gutenberg
import gutenbergpy.textget # Import the module that allows us to download the texts

dickens_id = 98 # ID for the book that we want to select.
dickens_raw = gutenbergpy.textget.get_text_by_id(dickens_id) # book, with header and license
dickens_clean = gutenbergpy.textget.strip_headers(dickens_raw) # removes headers and license

dickens = dickens_clean.decode('utf-8') # matches encoding to ensure clean text
print(dickens[:200]) # prints first 200 characters of the text

We can start with counting up the total number of words in the text and looking at the number of words that appear most frequently. We'll use the `pandas` library and simply tally the number of times that individual words appear. 

In [ ]:
# We'll create a function that will count the words in a given text
import re
from collections import Counter
import pandas as pd

def word_counts(text):
    """Return a DataFrame of unique words and their counts, most frequent first."""
    # gutenbergpy returns bytes, so decode if needed
    if isinstance(text, bytes):
        text = text.decode("utf-8")

    # swap curly apostrophes for straight ones so "don’t" and "don't" count as the same word
    text = text.lower().replace("’", "'")

    # pull out words (keeping contractions like "don't")
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text)

    counts = Counter(words)
    df = pd.DataFrame(counts.most_common(), columns=["word", "count"])
    return df

In [ ]:
# Let's take a look at the top 20 most common words in A Tale of Two Cities
word_counts(dickens).head(20) # show the top 20 words in the list of words

In [17]:
# We'll create a function that removes stop words from a given text
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True) # downloads NLTK's stop word lists if needed

def remove_stop_words(text, stop_words=None):
    """Return the text as a lowercase string of words with stop words removed."""
    # gutenbergpy returns bytes, so decode if needed
    if isinstance(text, bytes):
        text = text.decode("utf-8")

    # use NLTK's English stop word list unless we pass in our own
    if stop_words is None:
        stop_words = set(stopwords.words('english'))

    # swap curly apostrophes for straight ones so contractions match the stop word list
    text = text.lower().replace("\u2019", "'")

    # split into words the same way word_counts does, then drop the stop words
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text)
    kept = [word for word in words if word not in stop_words]

    return " ".join(kept) # join back into a string so it can go into word_counts

In [18]:
# Now let's look at the top 20 words in A Tale of Two Cities without the stop words
dickens_no_stops = remove_stop_words(dickens)
word_counts(dickens_no_stops).head(20)

,word,count
0,said,661
1,mr,622
2,one,438
3,would,343
4,lorry,339
5,upon,289
6,man,284
7,could,282
8,defarge,282
9,little,267


### Tagging the Book with spaCy
Counting words tells us *what* words appear, but not *how* they're being used. Part-of-speech (POS) tagging labels each word with its grammatical role (noun, verb, pronoun, etc.), and named entity recognition (NER) picks out names of people, places, organizations, and so on.

We'll use `spaCy` for this. Note that we run spaCy on the **original** text, not the stop-word-free version: the tagger relies on the surrounding words (including stop words) and capitalization to decide what each word is doing.

In [66]:
# We'll create a function that runs spaCy over a whole book
import spacy

# load spaCy's small English model (we don't need the dependency parser, so we turn it off to save time)
nlp = spacy.load("en_core_web_sm", disable=["parser"])

def tag_text(text):
    """Run spaCy over the text one paragraph at a time and return a list of processed Docs."""
    # gutenbergpy returns bytes, so decode if needed
    if isinstance(text, bytes):
        text = text.decode("utf-8")

    # split on blank lines into paragraphs, and join the line breaks inside each paragraph
    # (spaCy can't take a whole book at once, and paragraphs are a natural unit anyway)
    paragraphs = [" ".join(p.split()) for p in re.split(r"\n\s*\n", text) if p.strip()]

    return list(nlp.pipe(paragraphs, batch_size=50))

In [67]:
# Tag A Tale of Two Cities (this takes a minute or so, since spaCy is reading the whole book)
dickens_docs = tag_text(dickens)

In [68]:
# We'll create a function that puts every word and its tags into a DataFrame
def pos_table(docs):
    """Return a DataFrame with one row per word: the word, its part of speech, its detailed tag, and its lemma."""
    rows = []
    for doc in docs:
        for token in doc:
            if token.is_alpha: # skip punctuation, numbers, and whitespace
                rows.append({
                    "word": token.text,
                    "pos": token.pos_, # broad part of speech (NOUN, VERB, PRON, ...)
                    "tag": token.tag_, # more detailed tag (e.g., VBD = past tense verb)
                    "lemma": token.lemma_,
                })
    return pd.DataFrame(rows)

dickens_pos = pos_table(dickens_docs)
dickens_pos.head(20)

,word,pos,tag,lemma
0,A,DET,DT,a
1,TALE,NOUN,NN,tale
2,OF,ADP,IN,of
3,TWO,NUM,CD,two
4,CITIES,NOUN,NNS,city
5,A,DET,DT,a
6,STORY,NOUN,NN,story
7,OF,ADP,IN,of
8,THE,DET,DT,the
9,FRENCH,PROPN,NNP,FRENCH


### Counting Parts of Speech and Places
Now that every word is tagged, we can count just the words in a particular category. spaCy uses the [Universal POS tags](https://universaldependencies.org/u/pos/): `PRON` for pronouns, `VERB` for verbs, `NOUN` for common nouns, `PROPN` for proper nouns, and so on. Note that spaCy tags helping verbs like *was* and *had* (as in "had gone") as `AUX` rather than `VERB`.

For places, we'll use the entity labels `GPE` (countries, cities, states), `LOC` (other locations, like rivers or mountains), and `FAC` (buildings, bridges, streets).

In [69]:
# We'll create a function that counts the words with a given part of speech
def pos_counts(docs, pos):
    """Return a DataFrame of words with the given part(s) of speech and their counts, most frequent first."""
    # accept either a single tag ("NOUN") or a list of tags (["NOUN", "PROPN"])
    if isinstance(pos, str):
        pos = [pos]

    words = [token.text.lower() for doc in docs for token in doc
             if token.pos_ in pos and token.is_alpha]

    counts = Counter(words)
    return pd.DataFrame(counts.most_common(), columns=["word", "count"])

In [70]:
# Pronouns in A Tale of Two Cities
pos_counts(dickens_docs, "PRON").head(20)

,word,count
0,it,2060
1,his,2009
2,i,1957
3,he,1855
4,you,1420
5,her,1043
6,him,972
7,my,655
8,they,567
9,me,528


In [71]:
# Verbs in A Tale of Two Cities
pos_counts(dickens_docs, "VERB").head(20)

,word,count
0,said,658
1,know,231
2,see,197
3,looked,194
4,made,186
5,say,165
6,come,157
7,came,141
8,go,139
9,looking,126


In [72]:
# Nouns in A Tale of Two Cities
pos_counts(dickens_docs, "NOUN").head(20)

,word,count
0,man,297
1,time,262
2,hand,246
3,night,216
4,day,204
5,way,184
6,father,183
7,face,181
8,head,176
9,eyes,163


In [73]:
# We'll create a function that counts the named entities with given labels (by default, places)
def entity_counts(docs, labels=("GPE", "LOC", "FAC")):
    """Return a DataFrame of named entities with the given labels and their counts, most frequent first."""
    entities = [ent.text for doc in docs for ent in doc.ents if ent.label_ in labels]

    counts = Counter(entities)
    return pd.DataFrame(counts.most_common(), columns=["entity", "count"])

# Places in A Tale of Two Cities
entity_counts(dickens_docs).head(20)

,entity,count
0,Paris,62
1,France,58
2,England,38
3,Sydney,32
4,Monseigneur,32
5,London,25
6,Bastille,21
7,Stryver,20
8,Soho,16
9,Dover,15


Take a look through these tables: does the tagger get everything right? Models like spaCy's are trained mostly on modern text (news, web pages), so 19th-century prose can trip them up. Watch for words tagged with the wrong part of speech, and for "places" that are actually people (or vice versa).

### Lemmatization
Lemmatization reduces each word to its dictionary form (its *lemma*): *was* and *were* become *be*, *hands* becomes *hand*, *said* becomes *say*. spaCy already worked out each word's lemma when it tagged the book, so we just need to collect them. We'll join the lemmas back into a single string so that it can be fed into `remove_stop_words` and `word_counts`, just like the original text.

In [30]:
# We'll create a function that turns tagged text into a string of lemmas
def lemmatize(docs):
    """Return the text as a lowercase string of lemmas, ready for remove_stop_words or word_counts."""
    lemmas = [token.lemma_.lower() for doc in docs for token in doc if token.is_alpha]
    return " ".join(lemmas)

dickens_lemmas = lemmatize(dickens_docs)
print(dickens_lemmas[:200])

a tale of two city a story of the french revolution by charles dickens content book the first recall to life chapter i the period chapter ii the mail chapter iii the night shadows chapter iv the prepa


In [31]:
# Compare these to the top 20 words (without stop words) from before
word_counts(remove_stop_words(dickens_lemmas)).head(20)

,word,count
0,say,869
1,look,479
2,one,442
3,go,416
4,see,413
5,know,407
6,man,392
7,hand,374
8,lorry,369
9,come,361


### Putting It All Together
Now we can combine everything into a single workflow: download a book, tag it once with spaCy, and build all of our tables from that one pass. The function returns a dictionary of DataFrames, so we can look at whichever table we want by name.

In [32]:
# We'll create a function that runs the whole workflow on a Project Gutenberg book
def analyze_book(book_id):
    """Download a Project Gutenberg book and return a dictionary of word frequency tables."""
    # 1. download the book and strip the Project Gutenberg header and license
    raw = gutenbergpy.textget.get_text_by_id(book_id)
    text = gutenbergpy.textget.strip_headers(raw).decode("utf-8")

    # 2. tag the book with spaCy (the slow step, so we only do it once)
    docs = tag_text(text)

    # 3. build each table from the original text or the tagged docs
    return {
        "words": word_counts(text),
        "words_no_stops": word_counts(remove_stop_words(text)),
        "lemmas_no_stops": word_counts(remove_stop_words(lemmatize(docs))),
        "pronouns": pos_counts(docs, "PRON"),
        "verbs": pos_counts(docs, "VERB"),
        "nouns": pos_counts(docs, "NOUN"),
        "places": entity_counts(docs),
    }

In [33]:
# Let's try it out on Moby-Dick
moby = analyze_book(2701)

# show the top 10 of each table side by side
pd.concat({name: table.head(10) for name, table in moby.items()}, axis=1)

words        words_no_stops       lemmas_no_stops       pronouns        \
   word  count           word count            word count     word count   
0   the  14535          whale  1110           whale  1543      his  2529   
1    of   6624            one   912             one   941       it  2489   
2   and   6447           like   647             man   774        i  2066   
3     a   4747           upon   566            like   650       he  1880   
4    to   4627            man   479            ship   619      him  1060   
5    in   4184            sea   454             say   610      you   891   
6  that   3085            old   450             see   578     they   666   
7   his   2532           ahab   440            upon   566       me   622   
8    it   2522           ship   439             sea   534    their   620   
9     i   2127             ye   431            ahab   504       my   587   

    verbs         nouns                 places        
     word count    word count           entity count  
0    said   296   whale   879           Pequod   132  
1  seemed   283    ship   508            Flask    73  
2     see   256     man   506        Nantucket    62  
3     say   229     sea   429           Bildad    56  
4      go   191    boat   332             thou    55  
5    made   177    head   330        Leviathan    42  
6    come   173    time   327  the White Whale    32  
7    seen   164     way   267            Jonah    31  
8     let   158  whales   243          Pacific    29  
9    look   157     men   239        Greenland    27

### Comparing Books: Pronouns
Pronouns are a good example of why we might *not* want to throw stop words away: they tell us who a book is about and who is speaking. Let's put the top 20 pronouns from *A Tale of Two Cities* and *Moby-Dick* side by side.

There's one catch: *Moby-Dick* is a much longer book (about 217,000 words versus about 137,000), so its raw counts will be bigger just because there are more words. To compare the two fairly, we'll also convert each count into a rate: how many times the pronoun appears per 1,000 words.

In [ ]:
# We'll create a function that adds a "per 1,000 words" rate to a table of counts
def per_1000_words(counts, words, n=20):
    """Return the top n rows of a counts table with each count converted to a rate per 1,000 words in the book."""
    total_words = words["count"].sum() # total number of words in the book
    table = counts.head(n).copy()
    table["per_1000_words"] = (table["count"] / total_words * 1000).round(1)
    return table

# Top 20 pronouns in A Tale of Two Cities and Moby-Dick, side by side
pd.concat({
    "A Tale of Two Cities": per_1000_words(pos_counts(dickens_docs, "PRON"), word_counts(dickens)),
    "Moby-Dick": per_1000_words(moby["pronouns"], moby["words"]),
}, axis=1)

What do these lists tell us about the two books? A few things to look for:
- **Who's in the book?** "She" doesn't make the top 20 for *Moby-Dick* at all, and "her" appears about five times less often than in *A Tale of Two Cities* (1.5 versus 7.6 times per 1,000 words). And many of the "her"s in *Moby-Dick* aren't women at all, but ships and boats.
- **Who's speaking?** "We" and "us" show up in *Moby-Dick*'s list but not in Dickens', which hints at a narrator who is one member of a crew.
- **How much is story, and how much is something else?** *Moby-Dick* uses fewer pronouns overall. Long stretches of the book are essays about whales and whaling rather than scenes between characters.
- **What did the tagger miss?** *Moby-Dick* is full of "thou," "thee," and "ye," but you won't find "thou" anywhere in its pronoun list (spaCy tagged it as something else, and it even showed up in our list of places). A model trained on modern English doesn't always recognize older forms, so its mistakes shape our counts.

## Your Turn

In [ ]:
# Now you can grab a book of your choice
import gutenbergpy.textget # Import the module that allows us to download the texts

book_id = 0 # Replace 0 with the ID number of the book you want (e.g., 1342 for Pride and Prejudice)
book_raw = gutenbergpy.textget.get_text_by_id(book_id) # book, with header and license
book_clean = gutenbergpy.textget.strip_headers(book_raw) # removes headers and license

book = book_clean.decode('utf-8') # matches encoding to ensure clean text
print(book[:200]) # prints first 200 characters of the text